# 第 4 课作业：把 next state 和 register state 分开

这份作业对应 **第 4 课：电路怎样记住上一时刻的膜电位？**

这一题最重要的不是“把列表跑出来”，而是始终知道：**边沿前 register 里是什么、组合逻辑算出了什么、边沿后真正保存了什么。**

## 本题目标

你要分别实现：

- combinational_step()：计算 candidate、spike、value_to_store；
- clock_edge()：把已经决定好的 value_to_store 写成新的 register state；
- run_clocked_accumulator()：把前两步按 cycle 串起来，并记录 state/spike history。

states 包含初始 register value，所以长度必须比 inputs 多 1。

## 先不用代码：填一张 cycle 表

给定：

- initial_state = 1
- inputs = [2, 2, 1]
- threshold = 4
- reset = 0

先手工填：

| cycle | state before | input | candidate | spike? | value_to_store | state after edge |
|---|---:|---:|---:|---|---:|---:|
| 0 | 1 | 2 | ? | ? | ? | ? |
| 1 | ? | 2 | ? | ? | ? | ? |
| 2 | ? | 1 | ? | ? | ? | ? |

完成后检查自己有没有把 candidate 和 state after edge 写成同一个概念。

## Part A：组合逻辑

### 这个函数做什么？

`combinational_step()` 表示一个 clock edge 到来之前，组合逻辑根据“当前 register state + 当前输入”算出的决定。它**不负责真正更新 register**。

### 输入

- `state`：当前 edge 之前，register 中保存的旧状态；
- `input_value`：本 cycle 的输入；
- `threshold`：触发 spike 的阈值；
- `reset`：如果 spike，下一次 edge 要写入 register 的 reset 值。

### 输出

函数返回 **三个值**，顺序固定为：

`(candidate_state, spike, value_to_store)`

其中：

1. `candidate_state`：当前 state 加上 input 后得到的候选值，尚未经过 reset；
2. `spike`：布尔值，表示这个 candidate 是否达到或超过 threshold；
3. `value_to_store`：如果下一次 clock edge 现在发生，register 应该写入的值；spike 时是 `reset`，否则是 `candidate_state`。

这里得到的只是“组合逻辑决定”，register 本身还没有改变。

In [ ]:
def combinational_step(
    state: int,
    input_value: int,
    threshold: int,
    reset: int = 0,
) -> tuple[int, bool, int]:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: compute combinational next-state logic")
    # YOUR CODE ENDS HERE

    return candidate_state, spike, value_to_store

## Part B：clock edge

### 这个函数做什么？

`clock_edge()` 表示 clock edge 真正到来时，register 接受已经由组合逻辑决定好的 `value_to_store`。

### 输入

- `state`：edge 到来前 register 中的旧状态；
- `value_to_store`：组合逻辑已经决定好的、应该写入的新值。

### 输出

函数只返回 **一个整数**：

- edge 之后 register 中保存的新 state。

这里保留 `state` 参数，是为了让函数调用显式表达“edge 前的旧 state”。但这一步不再根据旧 state 重新计算新值——组合逻辑已经把结果放在 `value_to_store` 中。因此正确实现里 `state` 可以不参与计算，这不是漏写。

In [ ]:
def clock_edge(state: int, value_to_store: int) -> int:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: update the register at the clock edge")
    # YOUR CODE ENDS HERE

## Part C：把多个 cycle 串起来

### 这个函数做什么？

`run_clocked_accumulator()` 把 Part A 的组合逻辑和 Part B 的 clock edge 连续运行多个 cycle，并保存完整历史。

每个 `input` 对应一个 cycle：先算 combinational result，再发生一次 clock edge。

### 输入

- `inputs`：按时间顺序排列的输入列表；每个元素对应一个 cycle；
- `threshold`：所有 cycle 共用的 threshold；
- `reset`：spike 后写入的 reset；
- `initial_state`：第 0 个 cycle 开始前，register 的初始值。

### 输出

函数返回 **两个列表**，顺序固定为：

`(states, spikes)`

其中：

1. `states`：register state 历史。第一个元素是 `initial_state`，之后每个元素都是一次 clock edge 后真正保存的 state，所以长度应为 `len(inputs) + 1`；
2. `spikes`：每个 cycle 是否产生 spike 的布尔值列表，所以长度应为 `len(inputs)`。

`states[i]` 表示某个时刻 register 里真正保存的值，不是中间的 candidate。

In [ ]:
def run_clocked_accumulator(
    inputs: list[int],
    threshold: int,
    reset: int = 0,
    initial_state: int = 0,
) -> tuple[list[int], list[bool]]:
    state = initial_state
    states = [state]
    spikes = []

    for input_value in inputs:
        candidate_state, spike, value_to_store = combinational_step(
            state, input_value, threshold, reset
        )

        # candidate_state belongs to this cycle's combinational result.
        # states records register values after clock edges.
        # YOUR CODE STARTS HERE
        raise NotImplementedError("TODO: apply the clock edge and record the cycle")
        # YOUR CODE ENDS HERE

    return states, spikes

## 检查你的实现

先运行三个 Part，再调用 grader。

In [ ]:
# Course infrastructure: make the repository root importable from a notebook subdirectory.
from pathlib import Path
import sys

_repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "exercises" / "grader").is_dir()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from exercises.grader.lesson04 import check

check(
    combinational_step=combinational_step,
    clock_edge=clock_edge,
    run_clocked_accumulator=run_clocked_accumulator,
    language="zh",
)

## Human Check

1. 回到你手算的表：哪个列是 combinational result，哪个列才是 register state？
2. 当 candidate 达到 threshold 时，candidate 与 state after edge 为什么可能分别是 threshold 附近的值和 reset？
3. 你的 run_clocked_accumulator() 中，哪一步模拟 clock edge？
4. 为什么 states 要把 initial_state 也保留下来？